## Keys — Primary, Foreign, Natural, and Surrogate

Keys are the columns (or combinations of columns) that **identify and connect** rows. Understanding them is essential for joins, deduplication, and maintaining data integrity.

> **Note:** All data in this module is **entirely synthetic** and does not represent any real schools, pupils, or individuals.

### Types of key

| Key type | Definition | Example |
| --- | --- | --- |
| **Primary key** | A column (or set of columns) that **uniquely identifies** every row in a table. No nulls, no duplicates. | A system-generated `id` column, or a composite of `pupil_id` + `school_urn` |
| **Natural key** | A key derived from **real-world attributes** rather than a system-generated value. Meaningful to humans. | `national_insurance_number`, `urn`, `uln` |
| **Surrogate key** | A **system-generated identifier** with no real-world meaning, used purely to uniquely identify rows. Typically an auto-incrementing integer or UUID. | An `id` column populated by `ROW_NUMBER()` or `UUID()` |
| **Foreign key** | A column in one table that **references the primary key** of another table, creating a relationship between them. | `school_urn` in a pupil table referencing `school_urn` in a school table |

> **Note:** These categories are not mutually exclusive. A **natural key can also be the primary key** — and often is when the table has no surrogate. For example, if a table’s grain is one row per pupil per school, the composite natural key `(pupil_id, school_urn)` is also the primary key. The distinction is about *origin* (real-world attribute vs system-generated) rather than *role* (uniquely identifying rows).

### How keys relate to cardinality

The **primary key defines the grain**. If your primary key is `(pupil_id, school_urn)`, then one row = one pupil at one school. If you find two rows with the same primary key values, you have a duplicate.

### Natural keys vs surrogate keys

Natural keys are intuitive but can be **unstable** — a school’s URN might change after an academy conversion, or a pupil’s name might be updated. Surrogate keys are stable but meaningless to humans. Many systems use both: a surrogate key for internal relationships and natural keys for human-readable lookups.

### Foreign keys and joins

When joining tables, you connect a **foreign key** in one table to the **primary key** of another. Understanding the cardinality on each side is critical:

* **One-to-one**: each pupil has exactly one record in the demographics table
* **One-to-many**: each school has many pupils
* **Many-to-many**: pupils can attend multiple schools, and schools have multiple pupils (requires a linking table)

We’ll use `pupils_autumn_2024` and `schools_autumn_2024` from `catalog_40_copper_analyst_training.messy_data` to demonstrate.

In [0]:
-- Join pupils to schools using the foreign key relationship
-- school_urn in the pupils table references school_urn in the schools table
SELECT
  p.pupil_id
  ,p.first_name
  ,p.last_name
  ,p.school_urn
  ,s.school_name
  ,s.school_type
FROM catalog_40_copper_analyst_training.messy_data.pupils_autumn_2024 p
JOIN catalog_40_copper_analyst_training.messy_data.schools_autumn_2024 s
  ON p.school_urn = s.school_urn
ORDER BY p.pupil_id
LIMIT 15;

### Checking foreign key integrity

Joins only work correctly when the foreign key values in one table **actually exist** in the referenced table. When they don’t, you have **orphaned foreign keys** — rows that point to a record that isn’t there.

This can happen for many reasons:

* A school closed or was merged and removed from the reference table, but pupils still reference its URN
* A data load failed partway through, populating one table but not the other
* A manual data entry error introduced a URN that was never valid

Orphaned foreign keys are dangerous because they **silently drop rows** from inner joins. If you join pupils to schools and a pupil references a school that doesn’t exist, that pupil simply vanishes from your results — with no error or warning.

The pattern below uses a `LEFT JOIN` with a `WHERE ... IS NULL` filter to surface any orphaned records. In our synthetic data, pupil P019 (Sophie) references school URN 999999, which doesn’t appear in the schools table.

In [0]:
-- Find orphaned foreign keys: pupils referencing a school that doesn't exist
SELECT
  p.pupil_id
  ,p.first_name
  ,p.last_name
  ,p.school_urn as orphaned_school_urn
FROM catalog_40_copper_analyst_training.messy_data.pupils_autumn_2024 p
LEFT JOIN catalog_40_copper_analyst_training.messy_data.schools_autumn_2024 s
  ON p.school_urn = s.school_urn
WHERE s.school_urn IS NULL;

### Handling orphaned records — a business decision

Once you’ve identified orphaned foreign keys, you need to decide what to do with them. In some cases you may choose to **exclude these rows** from your analysis for data quality purposes — for example, if pupils reference schools that no longer exist in your reference data, including them could distort school-level aggregations or introduce misleading “unknown” categories.

However, removing data is never a purely technical decision. It is a **business decision** that should be:

* **Documented clearly** — record *what* was removed, *why*, and *how many rows* were affected
* **Proportionate** — if orphaned rows represent a significant share of the data, exclusion could introduce bias
* **Reversible** — filter rows out rather than deleting them, so the decision can be revisited
* **Communicated** — stakeholders should know that certain records were excluded and understand the impact

> **Good practice:** Add a comment or markdown cell to your notebook explaining the rationale whenever you filter out data. Future you (or a colleague) will thank you for it.

### Going further: discovering foreign key relationships dynamically

In practice, tables don’t always have formally declared foreign key constraints — especially in data lake environments. You can use `INFORMATION_SCHEMA` to discover columns that **share the same name** across tables, which are strong candidates for foreign key relationships.

The query below finds columns that appear in both the pupils and schools tables, suggesting a join relationship.

In [0]:
-- Find columns that share names between pupils and schools tables
-- These are strong candidates for foreign key / join relationships
SELECT
  p.column_name
  ,p.data_type as pupils_type
  ,s.data_type as schools_type
  ,CASE
    WHEN p.data_type = s.data_type THEN 'Types match - likely join key'
    ELSE 'Type mismatch - investigate'
  END as assessment
FROM (
  SELECT column_name, data_type
  FROM catalog_40_copper_analyst_training.information_schema.columns
  WHERE table_schema = 'messy_data' AND table_name = 'pupils_autumn_2024'
) p
JOIN (
  SELECT column_name, data_type
  FROM catalog_40_copper_analyst_training.information_schema.columns
  WHERE table_schema = 'messy_data' AND table_name = 'schools_autumn_2024'
) s ON p.column_name = s.column_name;

In [0]:
-- Dynamically check referential integrity for all shared columns
-- For each shared column, count how many values in pupils don't exist in schools

DECLARE integrity_sql STRING;

SET VAR integrity_sql = (
  WITH shared_cols AS (
    SELECT p.column_name
    FROM (
      SELECT column_name FROM catalog_40_copper_analyst_training.information_schema.columns
      WHERE table_schema = 'messy_data' AND table_name = 'pupils_autumn_2024'
    ) p
    JOIN (
      SELECT column_name FROM catalog_40_copper_analyst_training.information_schema.columns
      WHERE table_schema = 'messy_data' AND table_name = 'schools_autumn_2024'
    ) s ON p.column_name = s.column_name
  )
  SELECT aggregate(
    collect_list(
      concat(
        'SELECT ''', column_name, ''' as join_column, '
        ,'COUNT(DISTINCT p.`', column_name, '`) as distinct_values_in_pupils, '
        ,'SUM(CASE WHEN s.`', column_name, '` IS NULL THEN 1 ELSE 0 END) as orphaned_rows '
        ,'FROM catalog_40_copper_analyst_training.messy_data.pupils_autumn_2024 p '
        ,'LEFT JOIN (SELECT DISTINCT `', column_name, '` FROM catalog_40_copper_analyst_training.messy_data.schools_autumn_2024) s '
        ,'ON p.`', column_name, '` = s.`', column_name, '`'
      )
    )
    ,''
    ,(acc, x) -> CASE WHEN acc = '' THEN x ELSE concat(acc, ' UNION ALL ', x) END
  )
  FROM shared_cols
);

EXECUTE IMMEDIATE integrity_sql;